# [LAB-10] 로지스틱회귀ㅣ05-로지스틱회귀의 가정 검정(실습코드).ipynb

## #01. 준비작업 

### 1. 패키지 참조 

In [1]:
# 라이브러리 기본 참조 
from jussam import load_data
from helpers import *
from pandas import DataFrame

import numpy as np

# 가정 검정을 위한 참조 
from statsmodels.api import add_constant, Logit
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import chi2

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 데이터 가져오기 
3단원(다중로지스틱회귀)의 전처리가 모두 완료된 데이터를 사용함

In [2]:
origin = load_data("pima_indians_diabetes_preprocessed")
origin.head()

📚 피마 인디언 당뇨병 데이터셋에 대한 전처리 버전 - 결측치 처리, 이상치 처리, 라벨링 (출처: 자체 작업)


,Pregnancies,Glucose,BloodPressure,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.000,72.000,33.600,0.627,50,1
1,1,85.000,66.000,26.600,0.351,31,0
2,8,183.000,64.000,23.300,0.672,32,1
3,1,89.000,66.000,28.100,0.167,21,0
4,0,137.000,40.000,43.100,2.288,33,1


### 3. 기준선 모형 적합 
가정 검정은 변수를 소거하기 전의 모형에 대해 수행한다 

In [3]:
fit = my_logit.fit_model(origin, y='Outcome')
my_logit.report_variables(fit, origin)

,종속변수,독립변수,B,베타std,표준오차,z,유의확률,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Outcome,DiabetesPedigreeFunction,0.961,0.319,0.306,3.135,0.002,2.613,1.433,4.764,0.959,1.043
1,Outcome,Pregnancies,0.118,0.396,0.033,3.527,0.000,1.125,1.054,1.201,0.687,1.455
2,Outcome,BMI,0.091,0.626,0.016,5.777,0.000,1.095,1.062,1.130,0.860,1.162
3,Outcome,Glucose,0.035,1.082,0.004,9.763,0.000,1.036,1.029,1.043,0.865,1.156
4,Outcome,Age,0.017,0.199,0.010,1.723,0.085,1.017,0.998,1.037,0.615,1.626
5,Outcome,BloodPressure,-0.009,-0.110,0.009,-1.035,0.301,0.991,0.975,1.008,0.804,1.243


In [ ]:
# BooldPressure(p=0.301), Age(p=0.085) 의 유의확률이 유의하지 않다. 
# 이 상태에서 변수를 지우지 않고, 가정 검정부터 수행한다. 

### 로짓 선형성 

### 기준선 모형을 통해 확인한다 
#### 1) 가정 검정은 변수를 하나도 지우지 않은 상태에서 시작한다 
이 상태에서 유의확률만 보고 소거하면 실제로는 효과가 있는 변수를 지우게 된다. \
사용 데이터 : 피마 인디언 당뇨병(전처리 완료본), 종속변수 Outcome

확인포인트 : 보고표의 유의확률 열을 보고, 0.05를 넘는 변수가 무엇인지 찾는다. 지우지 말고 이름만 기억해 둔다. 
#### 2) 직선을 가정하는 대상은 확률이 아니라 로짓이다
확률과 독립변수가 직선인 것이 아니다.(그건 S자 곡선) - 로짓과 독립변수가 직선 \
깨지면 계수 베타 하나로 효과를 표현할 수 없다 (구간마다 효과가 다르므로 )

### 로짓 선형성 검정 방법 -> Box-Tidwell 
#### 보조항 x, ln(x)를 넣어보고, 그 항이 유의하면 직선이 아니다
기존 모형에 보조항 x, ln(x)를 추가하고, 그 계수 y(감마)의 유의성을 본다. -> 파생변수를 생성한다는 의미 \

판정 : 보조항의 p >= 0.05 이면 로짓 선형성 충족 

반드시 지켜야 하는 규칙 \
규칙1. ln(x)는 양수에서만 정의 

규칙2. 이분형 변수는 검정하지 않는다.

검정은 변수마다 다로 수행하고, 나머지 독립변수는 함께 통제한 상태로 둔다. 

확인포인트 : 결과표의 유의확률 열만 본다. 0.05미만인 변수가 몇개이고, z의 절대값이 가장 큰 변수는 무엇인가? 

## #02. 로짓 선형성 검정 

### 1. 기준선 모형 검정 
아래 코드는 처방 후 재검정에도 똑같은 구조로 반복된다.\
바뀌는 것은 대상 데이터프레임과 검정할 변수 목록 뿐이다. 

In [5]:
xnames = list(origin.columns.drop("Outcome")) # 종속변수를 제외한 독립변수 목록 
rows = [] # 결과를 담을 리스트 

for col in xnames:
    bt = origin[xnames].copy() # 독립변수만 복사 
    base = bt[col] # 독립변수의 값 
    if base.min() <=0: # 최소값이 0 이하라면?
        base = base - base.min() + 1 # ln의 정의역(양수)을 맞추기 위한 평행이동 

    bt["보조항"] = base * np.log(base)  # x* ln(x) 보조항 추가 

    # 보조항을 포함해 다시 적합 
    bt_fit = Logit(origin["Outcome"], add_constant(bt)).fit(disp=0)

    # 결과표 구성 
    rows.append({
        "독립변수":col,
        "z": bt_fit.tvalues["보조항"], 
        "유의확률": bt_fit.pvalues["보조항"],
        "로짓선형성": bt_fit.pvalues["보조항"]>=0.05,
    })

DataFrame(rows).set_index("독립변수").round(4)

,z,유의확률,로짓선형성
독립변수,,,
Pregnancies,0.020,0.984,True
Glucose,-1.084,0.278,True
BloodPressure,-0.564,0.573,True
BMI,-2.224,0.026,False
DiabetesPedigreeFunction,-2.239,0.025,False
Age,-4.460,0.000,False


In [6]:
# Age, Dia, BMI 위배 
# 나머지 세 변수는 충족 
# 위배가 가장 심한 Age부터 처방한다. 

## # 위배의 처방 3종 
### 로짓의 모양에 따라 처방이 정해져 있다. 
- 제곱항 추가 
- 로그변환 
- 구간화(범주화 )

1) 중심화 먼저 
제곱항은 반드시 중심화(평균 빼기) 후에 만든다. \
그냥 제곱하면 원변수와 상관이 커져 VIF가 폭발한다.

2) 변환의 근거 
변환의 근거는 분포의 왜도가 아니라 로짓 선형성 검정 결과다

## # 처방1. 중심화 후 제곱항 추가 
### 나이의 효과를 곡선에 허용하면 직선으로 잡히지 않던 관계가 드러난다
중심화 하는 이유 : 원변수와 제곱항의 상고나을 낮춰 다중공선성을 막기 위해(VIF 단계에서 재확인)\
처방이 정당했는지 확인하는 지표 : 우도비 검정, AIC(낮을수록 좋음), Pseudo R^2 

## #03. 로짓 선형성 위배의 처방 

### 1. Age - 중심화 후 제곱항 추가 

In [7]:
# 중심화 기준값 ( 예측 시에도 이 값을 그대로 사용 )
age_mean = origin["Age"].mean()

df1 = origin.copy() 
df1["Age_c"] = origin["Age"] - age_mean # 중심화된 나이 
df1["Age_c2"] = df1["Age_c"] ** 2 # 중심화된 나이의 제곱
df1 = df1.drop(columns=["Age"]) # 원래의 Age는 제거 

fit1 = my_logit.fit_model(df1, y="Outcome") 
my_logit.report_variables(fit1, df1)

,종속변수,독립변수,B,베타std,표준오차,z,유의확률,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Outcome,DiabetesPedigreeFunction,0.912,0.303,0.310,2.939,0.003,2.488,1.355,4.569,0.956,1.046
1,Outcome,BMI,0.084,0.582,0.016,5.291,0.000,1.088,1.055,1.123,0.843,1.186
2,Outcome,Age_c,0.075,0.878,0.016,4.628,0.000,1.077,1.044,1.112,0.280,3.566
3,Outcome,Pregnancies,0.047,0.158,0.036,1.285,0.199,1.048,0.976,1.126,0.576,1.737
4,Outcome,Glucose,0.036,1.106,0.004,9.764,0.000,1.037,1.029,1.044,0.865,1.156
5,Outcome,BloodPressure,-0.013,-0.156,0.009,-1.425,0.154,0.988,0.971,1.005,0.798,1.252
6,Outcome,Age_c2,-0.003,-0.710,0.001,-4.431,0.000,0.997,0.995,0.998,0.423,2.362


### 2. 처방이 모형을 실제로 개선했는지 우도비 검정으로 확인 

In [8]:
lr_stat = 2 * (fit1.llf - fit.llf) # 우도비 통계량 
lr_df = 1  # 추가된 모수의 수 (Age_c2 하나 )
lr_p = 1 - chi2.cdf(lr_stat, lr_df) # 우도비에 대한 검정 수행 

print(f"기준선 모형 : Pseudo R^2 = {fit.prsquared:.4f}, AIC= {fit.aic:.2f}")
print(f"제곱항 모형 : Pseudo R^2 = {fit1.prsquared:.4f}, AIC = {fit1.aic:.2f}") 
print(f"\n우도비 검정 : X^2({lr_df}) = {lr_stat:.4f}, p-value={lr_p:.4e}")

기준선 모형 : Pseudo R^2 = 0.2780, AIC= 686.86
제곱항 모형 : Pseudo R^2 = 0.3017, AIC = 666.74

우도비 검정 : X^2(1) = 22.1225, p-value=2.5579e-06


Age_c(p<0.001), Age_c2(p<0.001) 모두 유의 \
Age_c2의 계수가 음수이므로 위로 볼록한 역U자다.\
Pseudo R^2 0.278 -> 0.302\
AIC 686.86 -> 666.74\
우도비 검정 p < 0.001

만약 제곱항 추가시 유의하지 않게 나왔다면 로그변환을 해야 하는 변수이다. 

### 3. 처방1. 이후 재검정 
아직 손대지 않은 연속형 변수만 다시 검정한다. \
한 변수의 수정이 다른 변수의 진단 결과를 바꾸기 때문 

In [9]:
# 아직 변환하지 않은 연속형 독립변수만 재검정 
check_cols = ["Pregnancies", "Glucose", "BloodPressure", "BMI", "DiabetesPedigreeFunction"] 

# 제곱항 모형의 독립변수 전체 (보조항을 넣을 때 함께 통제해야 한다)
xnames1 = list(df1.columns.drop("Outcome"))

rows = [] # 결과를 담을 리스트 

for col in check_cols:
    bt = df1[xnames1].copy() # 독립변수만 복사 
    base = bt[col] # 독립변수의 값 
    if base.min() <= 0: # 최소값이 0 이하라면 ?
        base = base - base.min() + 1

    bt["보조항"] = base * np.log(base) # x * ln(x) 보조항 추가 

    # 보조항을 포함해 다시 적합 
    bt_fit = Logit(df1["Outcome"], add_constant(bt)).fit(disp=0)

    # 결과표 구성 
    rows.append({
        "독립변수": col, 
        "z" : bt_fit.tvalues["보조항"], 
        "유의확률": bt_fit.pvalues["보조항"], 
        "로짓선형성": bt_fit.pvalues["보조항"] >= 0.05, 
    })
DataFrame(rows).set_index("독립변수").round(4)

,z,유의확률,로짓선형성
독립변수,,,
Pregnancies,0.447,0.655,True
Glucose,-1.131,0.258,True
BloodPressure,-0.493,0.622,True
BMI,-1.802,0.071,True
DiabetesPedigreeFunction,-2.737,0.006,False


### 처방1. 이후 재검정 결과 확인 
BMI는 0.026 -> 0.071 로 저절로 해소됐다.\
DiabetesPedigreeFunction은 0.025 -> 0.006 으로 더 심해졌다.


## # 처방 후 재검정, 처방2. 로그변환 

### 고칠때마다 다시 검정한다.
한 변수의 수정이 다른 변수의 진단 결과를 바꾸기 때문\

재검정 대상 : 아직 손대지 않은 연속형 변수들(이미 변환한 변수는 변환된 형태로 포함)\
한 변수를 잘못설정하면 남은 곡선 효고가 다른 변수로 번진다 -> 원인 변수를 고치면 함께 해소되기도 한다 \
-> 그래서 위배가 심한 변수부터 하나씩 고친다(후진소거에서 하나씩 제거한 것과 같은 원리)\

처방2. 로그변환\
최솟값이 0보다 크면 일반 log, 0을 포함하면 log1p\
모든 연속형 변수가 기준을 통화할때까지 처방 -> 재검정을 반복한다 

확인포인트: 재검정표에서 직접 고친 변수와 손대지 않은 변수의 유의확률이 각각 어떻게 움직였는지 비교한다 

### 4. DiabetesPedigreeFunction 
최소값이 0.078로 0보다 크므로 log1p가 아닌 일반 log를 사용한다 

In [10]:
df2 = df1.copy() 
df2["DPF_log"] = np.log(df1["DiabetesPedigreeFunction"]) # 로그변환
df2 = df2.drop(columns=["DiabetesPedigreeFunction"]) # 기존 컬럼은 제거 

fit2 = my_logit.fit_model(df2, y="Outcome")
my_logit.report_variables(fit2, df2)

,종속변수,독립변수,B,베타std,표준오차,z,유의확률,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Outcome,DPF_log,0.561,0.361,0.158,3.552,0.000,1.753,1.286,2.389,0.967,1.035
1,Outcome,BMI,0.083,0.575,0.016,5.218,0.000,1.087,1.054,1.122,0.844,1.185
2,Outcome,Age_c,0.076,0.900,0.016,4.729,0.000,1.079,1.046,1.114,0.281,3.559
3,Outcome,Pregnancies,0.047,0.157,0.037,1.274,0.203,1.048,0.975,1.125,0.576,1.735
4,Outcome,Glucose,0.036,1.109,0.004,9.762,0.000,1.037,1.029,1.044,0.869,1.151
5,Outcome,BloodPressure,-0.013,-0.155,0.009,-1.416,0.157,0.988,0.971,1.005,0.799,1.251
6,Outcome,Age_c2,-0.003,-0.727,0.001,-4.521,0.000,0.997,0.995,0.998,0.424,2.357


### 5. 처방2. 이후 재검정 

In [11]:
# 검정대상에 로그변환된 DPF_log를 넣는다. 
check_cols = ["Pregnancies", "Glucose", "BloodPressure", "BMI", "DPF_log"]

# 제곱항 모형의 독립변수 전체 (보조항을 넣을때 함께 통제해야 한다)
xnames2 = list(df2.columns.drop("Outcome"))

rows = [] # 결과를 담을 리스트 

for col in check_cols:
    bt = df2[xnames2].copy() # 독립변수만 복사 
    base = bt[col] # 독립변수의 값
    if base.min() <= 0 : # 최소값이 0 이하라면?
        base = base - base.min() + 1 # ln의 정의역(양수)을 맞추기 위한 평행이동

    bt["보조항"] = base * np.log(base) # x * ln(x) 보조항 추가 

    # 보조항을 포함해 다시 적합 
    bt_fit = Logit(df2["Outcome"], add_constant(bt)).fit(disp=0)

    # 결과표 구성 
    rows.append({
        "독립변수": col,
        "z": bt_fit.tvalues["보조항"],
        "유의확률": bt_fit.pvalues["보조항"], 
        "로짓선형성": bt_fit.pvalues["보조항"]>=0.05, 
    })

DataFrame(rows).set_index("독립변수").round(4)

,z,유의확률,로짓선형성
독립변수,,,
Pregnancies,0.450,0.653,True
Glucose,-1.117,0.264,True
BloodPressure,-0.459,0.646,True
BMI,-1.769,0.077,True
DPF_log,-0.612,0.541,True


## # 가정 충족한 뒤에 변수 선택 
### 변수 선택은 가정 검정을 통과한 모형에서 처음부터 다시 한다 \
가정이 깨진 상태의 후진소거 결과는 신뢰할 수 없다 : 폐기하고 재수행\
후진소거 : 유의하지 않은 변수를 한번에 하나씩 제거하며 반복\
나이를 곡선으로 제대로 통제하면 함께 움직이던 변수의 설명력이 나이쪽으로 흡수될 수 있다. \

주의\
제곱항이 있는 모형은 오즈비를 한 줄로 해석하지 않는다\
효과는 1차항과 2차항이 함께 만든다.\

확인포인트 \
제거된 변수와 최종 채택 변수를 확인하고, 가정 검정 전 결론과 무엇이 뒤집혔는지 비교한다. 

## #04. 가정을 충족한 상태에서 변수 선택 
### 1. 다중 공선성 해결 

In [12]:
df3 = my_prep.reduce_vif(df2, 
                         columns=list(df2.drop(columns="Outcome").columns))
df3.head()


완료! 남은변수 : ['Age_c', 'Age_c2', 'BMI', 'BloodPressure', 'DPF_log', 'Glucose', 'Pregnancies']
최대 VIF = 3.56


,Pregnancies,Glucose,BloodPressure,BMI,Outcome,Age_c,Age_c2,DPF_log
0,6,148.000,72.000,33.600,1,16.649,277.195,-0.467
1,1,85.000,66.000,26.600,0,-2.351,5.526,-1.047
2,8,183.000,64.000,23.300,1,-1.351,1.825,-0.397
3,1,89.000,66.000,28.100,0,-12.351,152.543,-1.790
4,0,137.000,40.000,43.100,1,-0.351,0.123,0.828


### 2. 후진 소거 적용 모델링 

In [13]:
fit3 = my_logit.auto_logit(df3, y="Outcome", backward=True, report=False, plot=False)

my_logit.report_variables(fit3, df3)


유의하지 않은 독립변수 제거 -> Pregnancies (p=0.2026)
유의하지 않은 독립변수 제거 -> BloodPressure (p=0.1595)


,종속변수,독립변수,B,베타std,표준오차,z,유의확률,오즈비(OR),OR 95% 하한,OR 95% 상한,공차,VIF
0,Outcome,DPF_log,0.560,0.360,0.157,3.561,0.000,1.751,1.287,2.384,0.973,1.028
1,Outcome,Age_c,0.083,0.982,0.013,6.665,0.000,1.087,1.061,1.114,0.495,2.022
2,Outcome,BMI,0.075,0.519,0.015,4.969,0.000,1.078,1.047,1.111,0.910,1.099
3,Outcome,Glucose,0.035,1.088,0.004,9.714,0.000,1.036,1.029,1.043,0.877,1.140
4,Outcome,Age_c2,-0.004,-0.789,0.001,-5.368,0.000,0.996,0.995,0.998,0.510,1.960


## # 관측치의 독립성 
# 독립성 가정은 로지스틱 회귀에도 그대로 남아있다. 
로지스틱 회귀에서는 원시잔차 대신 피어슨 잔차로 계산한다 

주의\
DW는 시계열 전용이므로 수집 순서에 의미가 없으면 참고만 하고\
반복 측정, 군집자료(같은 사람, 가족, 병원)는 DW가 정상이어도 독립성 위배다. 

독립성은 최종적으로 자료 수집 설계로 판단한다. 

확인포인트 : DW값이 1.5~2.5 구간 안에 들어오는지만 본다. 

## #05. 관측치의 독립성 
### 1. 피어슨 잔차를 사용한 독립성 검정 

In [14]:
dw = durbin_watson(fit3.resid_pearson) # 피어슨 잔차 사용 독립성 검정 

if 1.5 <= dw <=2.5:
    interpretation="잔차는 독립성을 만족함"
elif dw < 1.5:
    interpretation = "잔차에 양(+)의 자기상관이 존재할 가능성 있음( 독립성 위반 )"
else:
    interpretation = "잔차에 음(-)의 자기상관이 존재할 가능성 있음( 독립성 위반 )"

print(f"Durbin_Watson = {dw:.4f} ::: {interpretation}")

Durbin_Watson = 1.8879 ::: 잔차는 독립성을 만족함


## #06. 성능 평가 

In [15]:
my_logit.report_performance(fit3, plot=False)

,예측 0 (Negative),예측 1 (Positive)
실제 0 (Negative),419,56
실제 1 (Positive),103,146


,정확도(Accuracy),정밀도(Precision),"재현율(Recall, TPR)","위양성율(Fallout, FPR)","특이성(Specificity, TNR)",F1,AUC,AUC 판단,진단오즈비(DOR)
preformance,0.780,0.723,0.586,0.118,0.882,0.647,0.854,우수,10.606


## #07. 로짓 선형성 가정 모듈화 확인 

In [16]:
my_logit.test_linear(fit, origin )

,z,p-value,linearity,위치이동,result
독립변수,,,,,
Pregnancies,0.020,0.984,True,True,귀무가설 채택 -> 로짓 선형성 위배 근거 없음
Glucose,-1.084,0.278,True,False,귀무가설 채택 -> 로짓 선형성 위배 근거 없음
BloodPressure,-0.564,0.573,True,False,귀무가설 채택 -> 로짓 선형성 위배 근거 없음
BMI,-2.224,0.026,False,False,대립가설 채택 -> 로짓 선형성 위배(변환 필요)
DiabetesPedigreeFunction,-2.239,0.025,False,False,대립가설 채택 -> 로짓 선형성 위배(변환 필요)
Age,-4.460,0.000,False,False,대립가설 채택 -> 로짓 선형성 위배(변환 필요)
